# TCGA whole-slide images, from IDC's bucket to the store

This notebook walks the path one slide takes:

1. **IDC's index** says which slides exist and which bucket folder holds each one.
2. **The bucket** holds one DICOM file per pyramid level, named by UUID.
3. **Each DICOM header** describes the slide and its tiles; the tiles follow it, with no table of where each one starts.
4. **The store** `earthmover-biology/tcga` references every tile in place, under `tcga_kirc/<patient>/<slide>/image/`.

Parts 1 to 3 need only internet access, since the bucket and the index are public. Part 4 needs an Arraylake login with read access (`al auth login`). Run it with

```bash
uv run --package tcga-ingest --group notebook --with jupyterlab jupyter lab ingests/tcga/tcga.ipynb
```

In [ ]:
import time

import idc_index_data
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import s3fs
import zarr

SLIDE = "TCGA-B0-4712-01Z-00-DX1"
IDC_RELEASE = "https://github.com/ImagingDataCommons/idc-index-data/releases/download/25.0.5"

## 1. IDC's index

The index is a set of Parquet tables that IDC builds from the DICOM headers of everything it hosts. `idc_index.parquet` has one row per DICOM series and ships inside the `idc-index-data` package. `sm_index.parquet` adds slide-microscopy fields, one row per slide, and is a release asset on GitHub. Neither needs the `idc-index` client: pandas reads them directly.

In [ ]:
series = pd.read_parquet(idc_index_data.IDC_INDEX_PARQUET_FILEPATH)
slides = pd.read_parquet(f"{IDC_RELEASE}/sm_index.parquet")
print(f"{len(series):,} series in IDC, {len(slides):,} of them slides")

tcga = series[series.collection_id.str.startswith("tcga_") & (series.Modality == "SM")].merge(slides, on="SeriesInstanceUID")
dx = tcga[tcga.ContainerIdentifier.str.contains("-DX")]
print(f"TCGA: {len(tcga):,} slides; {len(dx):,} diagnostic (DX) slides from {dx.PatientID.nunique():,} patients, "
      f"{dx.series_size_MB.sum() / 1e6:.1f} TB")

(dx.groupby("collection_id")
   .agg(slides=("SeriesInstanceUID", "size"), patients=("PatientID", "nunique"), GB=("series_size_MB", lambda s: round(s.sum() / 1e3)))
   .sort_values("slides", ascending=False))

In [ ]:
row = dx[dx.ContainerIdentifier == SLIDE].iloc[0]
row[["collection_id", "PatientID", "ContainerIdentifier", "SeriesDescription", "ManufacturerModelName",
     "transfer_syntax_name", "min_PixelSpacing_2sf", "max_TotalPixelMatrixRows", "max_TotalPixelMatrixColumns",
     "series_size_MB", "license_short_name", "series_aws_url"]]

## 2. The slide in the bucket

`series_aws_url` is the slide's folder. Inside it, each file is one pyramid level or a thumbnail, named by a UUID. The bucket is public, so anonymous access works.

In [ ]:
fs = s3fs.S3FileSystem(anon=True)
folder = row.series_aws_url.rstrip("*")
files = sorted(fs.ls(folder, detail=True), key=lambda f: -f["size"])
for f in files:
    print(f"{f['size'] / 2**20:9.1f} MiB  {f['name']}")

## 3. The DICOM header

pydicom reads the header of the largest file, the base level, over S3 and stops before the pixels, so this fetches a few hundred kilobytes. The header carries nearly everything the index knows about the slide: who, what tissue, how it was prepared and scanned, and how its tiles are laid out.

In [ ]:
with fs.open(files[0]["name"], block_size=1 << 18) as f:
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    header_bytes = f.tell()

for key in ["ContainerIdentifier", "PatientID", "ClinicalTrialProtocolID", "ClinicalTrialSiteID", "SeriesDescription",
            "Manufacturer", "ManufacturerModelName", "ImageType", "DimensionOrganizationType",
            "Rows", "Columns", "TotalPixelMatrixRows", "TotalPixelMatrixColumns", "NumberOfFrames"]:
    print(f"{key:28} {ds.get(key)}")
print(f"{'TransferSyntaxUID':28} {ds.file_meta.TransferSyntaxUID.name}")
print(f"{'PixelSpacing (mm)':28} {ds.SharedFunctionalGroupsSequence[0].PixelMeasuresSequence[0].PixelSpacing}")
print(f"header ends at byte {header_bytes:,}")

In [ ]:
specimen = ds.SpecimenDescriptionSequence[0]
print("anatomy:", [a.CodeMeaning for a in specimen.PrimaryAnatomicStructureSequence])
for step in specimen.SpecimenPreparationSequence:
    for item in step.SpecimenPreparationStepContentItemSequence:
        name = item.ConceptNameCodeSequence[0].CodeMeaning
        if name in ("Tissue Fixative", "Embedding medium", "Using substance"):
            print(f"{name}: {item.ConceptCodeSequence[0].CodeMeaning}")

The pixels follow the header as one encapsulated sequence: an offset table, then one item per tile. Here the offset table is empty, so the only way to find tile *n* is to walk past tiles 0 to *n* - 1. That is why the ingest reads every file once, and why the store is worth having: it keeps the walk's result, the byte range of every tile.

In [ ]:
with fs.open(files[0]["name"], block_size=1 << 18) as f:
    f.seek(header_bytes)
    element = f.read(12)
    item = f.read(8)
print("PixelData tag:", element[:4].hex(), " value length:", element[8:].hex(), "(undefined: encapsulated)")
print("offset table item:", item[:4].hex(), " length:", int.from_bytes(item[4:], "little"), "bytes")

## 4. The store

The store groups slides by collection and patient. Each slide's `image` is an OME-Zarr multiscale image, and each Zarr chunk is one DICOM tile, read from IDC's bucket when you ask for it.

In [ ]:
from arraylake import Client

repo = Client().get_repo("earthmover-biology/tcga")
root = zarr.open_group(repo.readonly_session("main").store, mode="r")
kirc = root["tcga_kirc"]
print(dict(kirc.attrs))
patients = sorted(name for name, _ in kirc.groups())
print(f"{len(patients)} patients, from {patients[0]} to {patients[-1]}")

slide = kirc[f"{row.PatientID}/{SLIDE}"]
print(slide.tree())
slide.attrs["idc"]

In [ ]:
image = slide["image"]
for dataset in image.attrs["ome"]["multiscales"][0]["datasets"]:
    level = image[dataset["path"]]
    scale = dataset["coordinateTransformations"][0]["scale"]
    print(f"level {dataset['path']}: {level.shape} {level.dtype}, chunks {level.chunks}, {scale[1]:.3f} µm per pixel")

The coarsest level of a few slides, each read whole. Every tile is a ranged GET into IDC's bucket.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, patient in zip(axes.flat, patients[::17]):
    (name, group), = kirc[patient].groups()
    levels = group["image"]
    coarsest = levels[str(len(list(levels.array_keys())) - 1)]
    start = time.perf_counter()
    pixels = coarsest[:]
    ax.imshow(np.moveaxis(pixels, 0, -1))
    ax.set_title(f"{name}\n{coarsest.shape[1:]} in {time.perf_counter() - start:.1f} s", fontsize=9)
    ax.axis("off")
fig.tight_layout()

A full-resolution region of the slide from parts 1 to 3, about 1 mm across. Change `y` and `x` to move around; the thumbnail on the left shows where the region sits.

In [ ]:
base, coarsest = image["0"], image[str(len(list(image.array_keys())) - 1)]
y, x, size = base.shape[1] // 2, base.shape[2] // 2, 4096

start = time.perf_counter()
region = base[:, y : y + size, x : x + size]
print(f"{region.shape} from level 0 in {time.perf_counter() - start:.1f} s, "
      f"{np.prod(np.ceil(np.array([size, size]) / base.chunks[1:])):.0f} tiles")

factor = base.shape[1] / coarsest.shape[1]
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6), width_ratios=[1, 1.4])
left.imshow(np.moveaxis(coarsest[:], 0, -1))
left.add_patch(plt.Rectangle((x / factor, y / factor), size / factor, size / factor, fill=False, color="lime", lw=1.5))
left.set_title(SLIDE, fontsize=9)
right.imshow(np.moveaxis(region, 0, -1))
right.set_title(f"level 0, rows {y}:{y + size}, columns {x}:{x + size}", fontsize=9)
for ax in (left, right):
    ax.axis("off")
fig.tight_layout()